# Backload Matching - AISQL Notebook

Self-contained companion to the **Backload Matching Engine** demo page in the ORS Control App.

Cell 1 stands up the schema and INSERTs ~30 rows via plain `VALUES`. Subsequent cells run AISQL functions on top: `AI_FILTER`, `AI_CLASSIFY`, `AI_EXTRACT`, `AI_AGG`, and `SNOWFLAKE.CORTEX.COMPLETE`. The final cell shows the raw VROOM JSON we send to `OPENROUTESERVICE_APP.CORE.OPTIMIZATION`.

Prereq: `OPENROUTESERVICE_APP` deployed with a Germany region. Active warehouse with Cortex access.

In [ ]:
-- Cell 1: schema + 30-row self-contained seed
ALTER SESSION SET query_tag = '{"origin":"sf_sit-is-fleet","name":"oss-backload-matching","version":{"major":1,"minor":0},"attributes":{"is_quickstart":1,"source":"notebook"}}';

CREATE SCHEMA IF NOT EXISTS FLEET_INTELLIGENCE.BACKLOAD_MATCHING
  COMMENT = '{"origin":"sf_sit-is-fleet","name":"oss-backload-matching","version":{"major":1,"minor":0},"attributes":{"is_quickstart":1,"source":"notebook"}}';

USE SCHEMA FLEET_INTELLIGENCE.BACKLOAD_MATCHING;

CREATE OR REPLACE TABLE EXTERNAL_OFFERS_DEMO (
  OFFER_ID VARCHAR, SOURCE VARCHAR, PICKUP_CITY VARCHAR, DROPOFF_CITY VARCHAR,
  WEIGHT_KG NUMBER, PRODUCT VARCHAR, PRICE_EUR NUMBER, HAZMAT BOOLEAN, LISTING_TEXT VARCHAR
) COMMENT = '{"origin":"sf_sit-is-fleet","name":"oss-backload-matching","version":{"major":1,"minor":0},"attributes":{"is_quickstart":1,"source":"notebook"}}';

INSERT INTO EXTERNAL_OFFERS_DEMO VALUES
  ('OFF-000001','TIMOCOM','Cologne','Hamburg', 18000,'Steel coils',  2400,FALSE,'TIMOCOM Cologne -> Hamburg 18000 kg steel coils EUR 2400 urgent FTL'),
  ('OFF-000002','WTRANSNET','Frankfurt','Berlin', 9500,'Pallets (general)',1200,FALSE,'WTRANSNET Frankfurt -> Berlin 9500 kg pallets EUR 1200 24h pickup'),
  ('OFF-000003','TELEROUTE','Munich','Hannover',12000,'Beverages',   1800,FALSE,'TELEROUTE Munich -> Hannover 12000 kg beverages EUR 1800 standard'),
  ('OFF-000004','B2P','Dortmund','Stuttgart',   22000,'Plastic granulate',2100,FALSE,'B2P Dortmund -> Stuttgart 22000 kg plastic granulate EUR 2100'),
  ('OFF-000005','TIMOCOM','Bremen','Leipzig',    7500,'Furniture',     900,FALSE,'TIMOCOM Bremen -> Leipzig 7500 kg furniture EUR 900 LTL'),
  ('OFF-000006','WTRANSNET','Hamburg','Munich', 24000,'Bulk paper',   2800,FALSE,'WTRANSNET Hamburg -> Munich 24000 kg bulk paper EUR 2800 FTL'),
  ('OFF-000007','TELEROUTE','Berlin','Cologne', 15000,'Steel coils',  1950,FALSE,'TELEROUTE Berlin -> Cologne 15000 kg steel coils EUR 1950'),
  ('OFF-000008','B2P','Hannover','Frankfurt',    8200,'Beverages',   1100,FALSE,'B2P Hannover -> Frankfurt 8200 kg beverages EUR 1100 reefer not required'),
  ('OFF-000009','TIMOCOM','Stuttgart','Bremen', 13000,'Plastic granulate',1700,TRUE, 'TIMOCOM Stuttgart -> Bremen 13000 kg ADR class 3 EUR 1700 hazmat certified driver required'),
  ('OFF-000010','WTRANSNET','Leipzig','Dortmund',5500,'Pallets (general)',780,FALSE,'WTRANSNET Leipzig -> Dortmund 5500 kg pallets EUR 780 next-day'),
  ('OFF-000011','TELEROUTE','Cologne','Munich', 19500,'Furniture',   2300,FALSE,'TELEROUTE Cologne -> Munich 19500 kg furniture EUR 2300 white-glove'),
  ('OFF-000012','B2P','Frankfurt','Hamburg',    21000,'Bulk paper',   2550,FALSE,'B2P Frankfurt -> Hamburg 21000 kg bulk paper EUR 2550 FTL'),
  ('OFF-000013','TIMOCOM','Munich','Berlin',    16500,'Steel coils',  2050,FALSE,'TIMOCOM Munich -> Berlin 16500 kg steel coils EUR 2050'),
  ('OFF-000014','WTRANSNET','Dortmund','Leipzig',9800,'Beverages',   1300,FALSE,'WTRANSNET Dortmund -> Leipzig 9800 kg beverages EUR 1300 LTL'),
  ('OFF-000015','TELEROUTE','Bremen','Stuttgart',11200,'Plastic granulate',1500,TRUE,'TELEROUTE Bremen -> Stuttgart 11200 kg ADR class 6 EUR 1500 hazmat'),
  ('OFF-000016','B2P','Hamburg','Frankfurt',    14500,'Pallets (general)',1750,FALSE,'B2P Hamburg -> Frankfurt 14500 kg pallets EUR 1750 standard'),
  ('OFF-000017','TIMOCOM','Berlin','Hannover',  17500,'Furniture',   2100,FALSE,'TIMOCOM Berlin -> Hannover 17500 kg furniture EUR 2100 retail'),
  ('OFF-000018','WTRANSNET','Hannover','Cologne',6700,'Bulk paper',   850,FALSE,'WTRANSNET Hannover -> Cologne 6700 kg bulk paper EUR 850 LTL'),
  ('OFF-000019','TELEROUTE','Stuttgart','Munich',23000,'Steel coils',  2700,FALSE,'TELEROUTE Stuttgart -> Munich 23000 kg steel coils EUR 2700 FTL'),
  ('OFF-000020','B2P','Leipzig','Bremen',       12500,'Beverages',   1650,FALSE,'B2P Leipzig -> Bremen 12500 kg beverages EUR 1650 reefer needed'),
  ('OFF-000021','TIMOCOM','Cologne','Frankfurt',  4500,'Pallets (general)',650,FALSE,'TIMOCOM Cologne -> Frankfurt 4500 kg pallets EUR 650 LTL'),
  ('OFF-000022','WTRANSNET','Frankfurt','Stuttgart',10500,'Plastic granulate',1400,FALSE,'WTRANSNET Frankfurt -> Stuttgart 10500 kg plastic granulate EUR 1400'),
  ('OFF-000023','TELEROUTE','Munich','Bremen',  20500,'Furniture',   2450,FALSE,'TELEROUTE Munich -> Bremen 20500 kg furniture EUR 2450 standard'),
  ('OFF-000024','B2P','Dortmund','Berlin',       7800,'Bulk paper',  1050,FALSE,'B2P Dortmund -> Berlin 7800 kg bulk paper EUR 1050'),
  ('OFF-000025','TIMOCOM','Hamburg','Stuttgart',18500,'Steel coils', 2250,FALSE,'TIMOCOM Hamburg -> Stuttgart 18500 kg steel coils EUR 2250 FTL'),
  ('OFF-000026','WTRANSNET','Berlin','Munich',  16000,'Beverages',   2000,FALSE,'WTRANSNET Berlin -> Munich 16000 kg beverages EUR 2000 reefer'),
  ('OFF-000027','TELEROUTE','Hannover','Leipzig',5200,'Plastic granulate',730,TRUE,'TELEROUTE Hannover -> Leipzig 5200 kg ADR class 9 EUR 730 hazmat licensed only'),
  ('OFF-000028','B2P','Stuttgart','Frankfurt',  11800,'Pallets (general)',1600,FALSE,'B2P Stuttgart -> Frankfurt 11800 kg pallets EUR 1600 standard'),
  ('OFF-000029','TIMOCOM','Leipzig','Cologne',  19000,'Furniture',   2350,FALSE,'TIMOCOM Leipzig -> Cologne 19000 kg furniture EUR 2350'),
  ('OFF-000030','WTRANSNET','Bremen','Hannover', 6300,'Bulk paper',   820,FALSE,'WTRANSNET Bremen -> Hannover 6300 kg bulk paper EUR 820 LTL');

SELECT COUNT(*) AS OFFERS_LOADED FROM EXTERNAL_OFFERS_DEMO;

## 1. AI_FILTER - keep only ADR/hazmat-relevant offers

Useful when a trailer's `HAZMAT_CERT = TRUE` and we want ADR-eligible offers only.

In [ ]:
SELECT OFFER_ID, SOURCE, PICKUP_CITY, DROPOFF_CITY, LISTING_TEXT
FROM FLEET_INTELLIGENCE.BACKLOAD_MATCHING.EXTERNAL_OFFERS_DEMO
WHERE AI_FILTER('Does this freight listing require ADR / hazardous-materials certification?', LISTING_TEXT);

## 2. AI_CLASSIFY - assign each offer to a cargo category

In [ ]:
SELECT OFFER_ID, PRODUCT,
       AI_CLASSIFY(LISTING_TEXT,
                   ['heavy industrial','consumer goods','food and beverages','hazardous materials','paper and packaging']
       ):labels[0]::VARCHAR AS CATEGORY
FROM FLEET_INTELLIGENCE.BACKLOAD_MATCHING.EXTERNAL_OFFERS_DEMO
ORDER BY OFFER_ID
LIMIT 15;

## 3. AI_EXTRACT - parse free-text listings into structured fields

In [ ]:
SELECT OFFER_ID,
       AI_EXTRACT(LISTING_TEXT,
                  ['service level (FTL or LTL)','special handling notes','temperature requirements']
       ) AS PARSED
FROM FLEET_INTELLIGENCE.BACKLOAD_MATCHING.EXTERNAL_OFFERS_DEMO
ORDER BY OFFER_ID
LIMIT 10;

## 4. AI_AGG - one-paragraph corridor summary across all offers

In [ ]:
SELECT AI_AGG(LISTING_TEXT,
              'Summarise the freight market posted today in 3 short sentences. Focus on dominant lanes, average weights, and ADR demand.'
) AS MARKET_SUMMARY
FROM FLEET_INTELLIGENCE.BACKLOAD_MATCHING.EXTERNAL_OFFERS_DEMO;

## 5. SNOWFLAKE.CORTEX.COMPLETE - dispatcher rationale for one assignment

Same prompt the React page sends. Replace the placeholder values to test other matches.

In [ ]:
SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-sonnet-4-5',
  'You are a fleet dispatcher coach. In two short sentences, explain why trailer TR-0001 (idle in Cologne, home depot Copenhagen) is a good match for TIMOCOM offer OFF-000005 (Bremen -> Leipzig, 320 km empty leg, 7500 kg furniture). Mention empty km and direction-to-home if relevant.'
) AS DISPATCHER_RATIONALE;

## 6. Raw OPTIMIZATION JSON

Tiny VROOM challenge with 2 trailers and 4 jobs to confirm the function signature works in this account.

In [ ]:
SELECT VEHICLE, COST, GEOJSON IS NOT NULL AS HAS_GEOMETRY
FROM TABLE(OPENROUTESERVICE_APP.CORE.OPTIMIZATION(PARSE_JSON('
{
  "vehicles": [
    {"id":1,"profile":"driving-hgv","start":[6.9603,50.9375],"end":[12.5655,55.6759],"capacity":[24000],"skills":[1,2,3]},
    {"id":2,"profile":"driving-hgv","start":[8.6821,50.1109],"end":[18.0686,59.3293],"capacity":[22000],"skills":[1,2]}
  ],
  "jobs": [
    {"id":101,"location":[7.4653,51.5136],"amount":[12000],"skills":[1],"priority":100},
    {"id":102,"location":[9.7320,52.3759],"amount":[ 9000],"skills":[1],"priority":100},
    {"id":103,"location":[8.8017,53.0793],"amount":[18000],"skills":[2],"priority":10},
    {"id":104,"location":[9.9937,53.5511],"amount":[ 6500],"skills":[2],"priority":10}
  ],
  "options": {"g": true}
}
')))
WHERE VEHICLE IS NOT NULL
ORDER BY VEHICLE;